# Step 5: A/B test design and simulation
**Proposal:** Shopify Magic suggests an image description when a merchant uploads a product image (merchant approves it).

**Question:** if Shopify ran an A/B test, how many merchants would it need, and would the analysis reliably detect a real effect?

We can't run the real test (only Shopify can randomise real merchants), so we:
1. take each store's **real baseline** from the research (share of images with descriptive alt text),
2. **size the test** from the real spread between stores,
3. **simulate** the experiment and analyse it exactly like a real readout,
4. repeat it 1,000 times to check the method's **power** and **false-alarm rate**.


In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import TTestIndPower

BASE = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "analysis" else os.getcwd()
CHARTS = os.path.join(BASE, "analysis", "charts"); os.makedirs(CHARTS, exist_ok=True)
rng = np.random.default_rng(42)
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRID,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlecolor": INK,
                     "axes.titlesize": 12, "font.size": 10, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "axes.axisbelow": True})


## 5.1 The primary metric, from real data
**Metric:** % of a store's product images with *descriptive* alt text (same 4-level rules as Step 3).
**Unit = the merchant**, because the feature is turned on per store and a store's images aren't independent.


In [ ]:
images = pd.read_csv(os.path.join(BASE, "data", "tables", "images_alt.csv"), keep_default_na=False)

def alt_class(alt, title):
    a = (alt or "").strip(); al = a.lower(); tl = (title or "").lower().strip()
    if a == "":
        return "missing"
    if (re.search(r"\.(jpe?g|png|webp|gif|avif)$", al)
            or re.search(r"(::|\|\||option\[|^img[_-]?\d|^dsc|pdp\d?|_main\b|^colou?r:|^size:|#colou?r_)", al)
            or re.search(r"^(featured|slideshow|image|img|photo|picture|thumbnail|banner|default|product)( ?\d+)?$", al)
            or not re.search(r"[A-Za-z]{3}", a) or (not re.search(r"\s", a) and re.search(r"[-_]", a))):
        return "junk"
    if al == tl or len(re.findall(r"[A-Za-z]+", a)) <= 3 or al in tl:
        return "minimal"
    return "descriptive"

images["descriptive"] = [alt_class(a, t) == "descriptive" for a, t in zip(images.alt, images.product_title)]
baseline = images.groupby("domain").descriptive.mean().mul(100)   # one value per store
BASE_MEAN, BASE_SD = baseline.mean(), baseline.std(ddof=1)
print(f"{len(baseline)} stores | baseline descriptive coverage: mean {BASE_MEAN:.1f}%, SD {BASE_SD:.1f} points")


## 5.2 Sample size: how many merchants per group?
Inputs: **alpha = 0.05** (5% false-alarm rate), **power = 0.80** (80% chance to detect a real effect),
and the **SD between stores** from the real data. Smaller effects need many more merchants.


In [ ]:
power_calc = TTestIndPower()
effects = np.arange(2, 16, 1)                           # improvement in percentage points
n_needed = [power_calc.solve_power(effect_size=d / BASE_SD, alpha=0.05, power=0.8) for d in effects]
size_table = pd.DataFrame({"detectable lift (pts)": effects, "merchants per group": np.ceil(n_needed).astype(int)})
display(size_table[size_table["detectable lift (pts)"].isin([3, 5, 10])])

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(effects, n_needed, color=BLUE, linewidth=2, marker="o", markersize=5)
ax.set_xlabel("Improvement to detect (percentage points)"); ax.set_ylabel("Merchants per group")
ax.set_title("Smaller effects need far more merchants", loc="left")
fig.tight_layout(); fig.savefig(os.path.join(CHARTS, "4_ab_sample_size.png"), dpi=200); plt.show()
N = int(np.ceil(power_calc.solve_power(effect_size=5 / BASE_SD, alpha=0.05, power=0.8)))
print(f"Plan: detect a 5-point lift -> {N:,} merchants per group")


## 5.3 Simulate one experiment
- Each simulated merchant starts from a **real store's baseline** (resampled from the 401 stores).
- Everyone drifts a little over 30 days (noise, SD 5 points).
- Treatment merchants get an **assumed true lift of 5 points** (an assumption, not a result), capped at 100%.
- **Guardrail:** time to publish a product (minutes). Assume treatment adds 2% (a small cost of reviewing suggestions).


In [ ]:
def simulate(n, true_lift, rng, guard_cost=0.02):
    ctrl0 = rng.choice(baseline.values, n); trt0 = rng.choice(baseline.values, n)
    ctrl = np.clip(ctrl0 + rng.normal(0, 5, n), 0, 100)
    trt = np.clip(trt0 + true_lift + rng.normal(0, 5, n), 0, 100)
    pub_c = rng.lognormal(np.log(12), 0.5, n)                    # minutes to publish a product
    pub_t = rng.lognormal(np.log(12 * (1 + guard_cost)), 0.5, n)
    return ctrl, trt, pub_c, pub_t, ctrl0, trt0

ctrl, trt, pub_c, pub_t, ctrl0, trt0 = simulate(N, 5, rng)


## 5.4 Analyse it like a real readout


In [ ]:
diff = trt.mean() - ctrl.mean()
t, p = stats.ttest_ind(trt, ctrl, equal_var=False)               # Welch's t-test (doesn't assume equal spread)
se = np.sqrt(trt.var(ddof=1) / N + ctrl.var(ddof=1) / N)
ci = (diff - 1.96 * se, diff + 1.96 * se)
guard_change = pub_t.mean() / pub_c.mean() - 1

print(f"Control:   {ctrl.mean():.1f}% descriptive   Treatment: {trt.mean():.1f}%")
print(f"Lift: {diff:+.1f} points   95% CI [{ci[0]:+.1f}, {ci[1]:+.1f}]   p = {p:.4f}")
print(f"Guardrail - time to publish: {guard_change:+.1%} (limit: +10%)")
decision = "SHIP" if (p < 0.05 and diff > 0 and guard_change < 0.10) else "DON'T SHIP"
print("Decision rule (set in advance):", decision)


### Why might a single test miss?
This one run may not reach p < 0.05 even though a real lift exists. Two reasons, both visible in the data:
1. **Huge spread between stores** (SD about 40 points): 31% of stores have 0% descriptive alt text and 11% already have 100%.
   That noise swamps a 5-point change.
2. **Ceiling effect:** stores already at 100% can't improve, so the *average* lift is smaller than 5 points.

**The fix: CUPED** (a standard variance-reduction method). We know each merchant's coverage **before**
the test. Adjusting for it removes the between-store spread, so we only compare *changes*.


In [ ]:
def cuped(y_c, y_t, x_c, x_t):
    """Adjust the outcome for the pre-test baseline (theta = cov(x, y) / var(x))."""
    x = np.r_[x_c, x_t]; y = np.r_[y_c, y_t]
    theta = np.cov(x, y)[0, 1] / np.var(x, ddof=1)
    adj_c = y_c - theta * (x_c - x.mean()); adj_t = y_t - theta * (x_t - x.mean())
    d = adj_t.mean() - adj_c.mean()
    se = np.sqrt(adj_t.var(ddof=1) / len(adj_t) + adj_c.var(ddof=1) / len(adj_c))
    return d, se, stats.ttest_ind(adj_t, adj_c, equal_var=False).pvalue

d2, se2, p2 = cuped(ctrl, trt, ctrl0, trt0)
print(f"Plain:  lift {diff:+.1f}  95% CI [{ci[0]:+.1f}, {ci[1]:+.1f}]  p = {p:.4f}")
print(f"CUPED:  lift {d2:+.1f}  95% CI [{d2-1.96*se2:+.1f}, {d2+1.96*se2:+.1f}]  p = {p2:.2g}")
print(f"CUPED shrinks the CI by {1 - se2/se:.0%}")
decision = "SHIP" if (p2 < 0.05 and d2 > 0 and guard_change < 0.10) else "DON'T SHIP"
print("Decision rule with CUPED:", decision)


**How to read this:** the CI is the range the true lift probably falls in. If it excludes 0 and the
guardrail stays under its limit, the pre-set rule says ship. Note the true lift was 5 by construction:
the point is to check that the *method* recovers it.


## 5.5 Does the method work? Repeat 1,000 times
- **Power:** with a real 5-point lift, how often does the test call it significant? (target about 80%)
- **False-alarm rate:** with **no** real lift, how often does it wrongly say significant? (target about 5%)


In [ ]:
def run_many(true_lift, reps=1000):
    r = np.random.default_rng(7); plain = []; cup = []; diffs = []
    for _ in range(reps):
        c, tr, _, _, c0, t0 = simulate(N, true_lift, r)
        plain.append(stats.ttest_ind(tr, c, equal_var=False).pvalue < 0.05)
        d, _, pc = cuped(c, tr, c0, t0); cup.append(pc < 0.05); diffs.append(d)
    return np.mean(plain), np.mean(cup), np.array(diffs)

pw_plain, pw_cuped, diffs_5 = run_many(5)
fa_plain, fa_cuped, diffs_0 = run_many(0)
print(pd.DataFrame({"plain t-test": [f"{pw_plain:.0%}", f"{fa_plain:.0%}"], "CUPED": [f"{pw_cuped:.0%}", f"{fa_cuped:.0%}"]},
                   index=["Power (real 5-pt lift)", "False alarms (no lift)"]))
print(f"Average measured lift with CUPED: {diffs_5.mean():.1f} pts (below 5 because of the ceiling at 100%)")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(diffs_0, bins=40, color=MUTED, alpha=0.6, label="No real effect")
ax.hist(diffs_5, bins=40, color=BLUE, alpha=0.8, label="Real 5-point lift")
ax.axvline(0, color=INK, linewidth=0.8)
ax.set_xlabel("CUPED-measured lift across 1,000 simulated experiments (points)"); ax.set_ylabel("Experiments")
ax.set_title("The test separates a real 5-point lift from no effect", loc="left"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(os.path.join(CHARTS, "5_ab_power_simulation.png"), dpi=200); plt.show()


## Takeaways (for the memo)
- The standard sample-size formula said about 1,000 merchants per group. The **simulation showed that's
  underpowered** with a plain t-test, because of the huge spread between stores and the 100% ceiling.
- **Using each merchant's pre-test coverage (CUPED)** fixes it: power rises well above 80% with the same
  sample, while false alarms stay near 5%.
- Lesson: check a formula's assumptions against the real data distribution before trusting it.
- **Limitation:** the 5-point lift and 2% guardrail cost are assumptions. Only the real test can measure them.
